In [ ]:
# ================================================================
# CELL 1 — Install packages
# ================================================================
# !pip install -q scikit-image opencv-python-headless


# ================================================================
# CELL 2 — Imports
# ================================================================
import cv2
import numpy as np
import matplotlib.pyplot as plt
from skimage.filters import frangi
import os
import time
import zipfile
import warnings
warnings.filterwarnings('ignore')

print('✅ All packages ready!')


# ================================================================
# CELL 3 — Define all functions (same as original)
# ================================================================

# ═══════════════════════════════════════════════════════════════
# FUNCTION 1: FOV Mask using Circle Fitting
# ═══════════════════════════════════════════════════════════════
def detect_fov_mask(image_bgr):
    """
    Detect the circular retinal disk and fit a PERFECT CIRCLE.
    This handles cases where background pixels are not exactly 0
    (e.g., value=6), which breaks simple thresholding.

    Returns: uint8 mask (255=inside disk, 0=background)
    """
    H, W = image_bgr.shape[:2]

    rgb_sum = image_bgr.astype(np.float32).sum(axis=2)
    rgb_sum_uint8 = np.clip(rgb_sum / 3, 0, 255).astype(np.uint8)

    _, mask_rough = cv2.threshold(rgb_sum_uint8, 20, 255, cv2.THRESH_BINARY)

    k = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (40, 40))
    mask_closed = cv2.morphologyEx(mask_rough, cv2.MORPH_CLOSE, k, iterations=5)

    n, labels, stats, _ = cv2.connectedComponentsWithStats(mask_closed, connectivity=8)
    if n > 1:
        largest = 1 + np.argmax(stats[1:, cv2.CC_STAT_AREA])
        mask_clean = (labels == largest).astype(np.uint8) * 255
    else:
        mask_clean = mask_closed

    contours, _ = cv2.findContours(
        mask_clean, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE
    )
    largest_contour = max(contours, key=cv2.contourArea)
    (cx, cy), radius = cv2.minEnclosingCircle(largest_contour)
    cx, cy, radius = int(cx), int(cy), int(radius)
    print(f'  FOV circle → center=({cx},{cy}), radius={radius}px')

    circle_mask = np.zeros((H, W), dtype=np.uint8)
    cv2.circle(circle_mask, (cx, cy), max(radius - 5, 1), 255, -1)

    return circle_mask


# ═══════════════════════════════════════════════════════════════
# FUNCTION 2: Green Channel Extraction
# ═══════════════════════════════════════════════════════════════
def extract_green_channel(image_bgr):
    """Extract green channel — best contrast for retinal vasculature."""
    return image_bgr[:, :, 1]


# ═══════════════════════════════════════════════════════════════
# FUNCTION 3: CLAHE Normalization
# ═══════════════════════════════════════════════════════════════
def brightness_normalize(gray):
    """
    CLAHE (Contrast Limited Adaptive Histogram Equalization).
    Standard brightness normalization for fundus images.
    Returns float32 in [0, 1].
    """
    clahe = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8, 8))
    return clahe.apply(gray).astype(np.float32) / 255.0


# ═══════════════════════════════════════════════════════════════
# FUNCTION 4: Frangi Vesselness Filter
# ═══════════════════════════════════════════════════════════════
def apply_frangi_vesselness(normalized_img,
                             sigmas=range(1, 6, 1),
                             alpha=0.5,
                             beta=0.5,
                             gamma=15):
    """
    Frangi vesselness filter — detects tubular vessel structures
    using Hessian matrix eigenvalue analysis at multiple scales.
    """
    vessel_map = frangi(
        normalized_img,
        sigmas=sigmas,
        alpha=alpha,
        beta=beta,
        gamma=gamma,
        black_ridges=True
    )
    return vessel_map


# ═══════════════════════════════════════════════════════════════
# FUNCTION 5: Post-processing (Invert + Mask + Contrast Stretch)
# ═══════════════════════════════════════════════════════════════
def postprocess_vessel_map(vessel_map, mask_bool):
    """
    1. Normalize using only pixels INSIDE the FOV
    2. Invert → vessels become DARK (black)
    3. Set background to pure WHITE
    4. Percentile contrast stretch for clean appearance
    """
    v = vessel_map.copy()

    v_in = v[mask_bool]
    v = np.clip((v - v_in.min()) / (v_in.max() - v_in.min() + 1e-8), 0, 1)

    v_inv = 1.0 - v
    v_inv[~mask_bool] = 1.0

    v_inside = v_inv[mask_bool]
    p2, p98 = np.percentile(v_inside, 2), np.percentile(v_inside, 98)
    v_inv[mask_bool] = np.clip((v_inside - p2) / (p98 - p2 + 1e-8), 0, 1)
    v_inv[~mask_bool] = 1.0

    return v_inv


# ═══════════════════════════════════════════════════════════════
# FUNCTION 6: Full Pipeline
# ═══════════════════════════════════════════════════════════════
def vessel_enhancement_pipeline(image_bgr,
                                 sigmas=range(1, 6, 1),
                                 alpha=0.5,
                                 beta=0.5,
                                 gamma=15):
    """
    Full pipeline:
      raw BGR
        → circle-fit FOV mask
        → green channel
        → CLAHE normalization
        → Frangi vesselness
        → invert + mask + contrast stretch
        → white bg, black vessels ✅
    """
    print('Running pipeline...')

    print('  [1/5] Detecting FOV mask (circle fit)...')
    fov_mask  = detect_fov_mask(image_bgr)
    mask_bool = fov_mask > 0

    print('  [2/5] Extracting green channel...')
    green = extract_green_channel(image_bgr)

    print('  [3/5] CLAHE normalization...')
    normalized = brightness_normalize(green)

    print('  [4/5] Frangi vesselness filter...')
    vessel_map = apply_frangi_vesselness(
        normalized, sigmas=sigmas, alpha=alpha, beta=beta, gamma=gamma
    )

    print('  [5/5] Post-processing (invert + mask + contrast stretch)...')
    vessel_final = postprocess_vessel_map(vessel_map, mask_bool)

    print('  ✅ Done!')
    return {
        'raw_bgr'     : image_bgr,
        'green'       : green,
        'normalized'  : normalized,
        'vessel_map'  : vessel_final,
        'fov_mask'    : fov_mask,
    }


# ═══════════════════════════════════════════════════════════════
# FUNCTION 7: Augmentation
# ═══════════════════════════════════════════════════════════════
def augment_pair(image, vessel_map,
                 flip=True,
                 max_rotation=15.0,
                 zoom_range=(0.9, 1.1),
                 brightness_range=(0.8, 1.2)):
    """
    Paper augmentations applied IDENTICALLY to image and vessel map:
      - Horizontal flip
      - Random rotation ±15°
      - Random zoom
      - Brightness/contrast jitter (image only)
    """
    h, w   = image.shape[:2]
    center = (w / 2, h / 2)

    def warp(img, M):
        return cv2.warpAffine(img, M, (w, h),
                              flags=cv2.INTER_LINEAR,
                              borderMode=cv2.BORDER_REFLECT)

    if flip and np.random.rand() > 0.5:
        image, vessel_map = cv2.flip(image, 1), cv2.flip(vessel_map, 1)

    angle = np.random.uniform(-max_rotation, max_rotation)
    Mr = cv2.getRotationMatrix2D(center, angle, 1.0)
    image, vessel_map = warp(image, Mr), warp(vessel_map, Mr)

    scale = np.random.uniform(*zoom_range)
    Mz = cv2.getRotationMatrix2D(center, 0.0, scale)
    image, vessel_map = warp(image, Mz), warp(vessel_map, Mz)

    alpha_b = np.random.uniform(*brightness_range)
    beta_b  = np.random.uniform(-0.1, 0.1)
    image   = np.clip(image * alpha_b + beta_b, 0, 1)

    return image, vessel_map


# ═══════════════════════════════════════════════════════════════
# FUNCTION 8: Visualization
# ═══════════════════════════════════════════════════════════════
def visualize_pipeline(results, title='', save_path=None):
    """Show 3-panel figure matching paper style."""
    fig, axes = plt.subplots(1, 3, figsize=(16, 5))
    fig.patch.set_facecolor('white')
    if title:
        fig.suptitle(title, fontsize=14, fontweight='bold', y=1.02)

    panels = [
        (results['raw_bgr'],    'rgb',  '(a) Raw Image'),
        (results['normalized'], 'gray', '(b) Normalized Image'),
        (results['vessel_map'], 'gray', '(c) Vessel Enhanced Map'),
    ]
    for ax, (img, cmap, label) in zip(axes, panels):
        if cmap == 'rgb':
            ax.imshow(cv2.cvtColor(img, cv2.COLOR_BGR2RGB))
        else:
            ax.imshow(img, cmap='gray', vmin=0, vmax=1)
        ax.set_title(label, fontsize=12, pad=8)
        ax.axis('off')

    plt.tight_layout()
    if save_path:
        plt.savefig(save_path, dpi=150, bbox_inches='tight', facecolor='white')
        print(f'Saved → {save_path}')
    plt.show()
    plt.close()


print('✅ All functions defined!')


# ================================================================
# CELL 4 — Configuration & Settings
# ================================================================

# ── Kaggle dataset path ──────────────────────────────────────────
# Adjust this to match your actual Kaggle dataset path
IMAGE_DIR  = '/kaggle/input/abu-sayed/Fundus_CIMT_2903 Dataset'

# ── Output directory ─────────────────────────────────────────────
OUTPUT_DIR = '/kaggle/working/vessel_output'
os.makedirs(OUTPUT_DIR, exist_ok=True)

# ── Frangi filter settings ───────────────────────────────────────
SIGMAS = range(1, 6, 1)
ALPHA  = 0.5
BETA   = 0.5
GAMMA  = 15

# ── Which outputs to save ────────────────────────────────────────
SAVE_VESSEL_MAP = True   # main output (white bg, black vessels)
SAVE_GREEN      = False
SAVE_NORMALIZED = False
SAVE_FOV_MASK   = False

VALID_EXTENSIONS = {'.jpg', '.jpeg', '.png', '.bmp', '.tif', '.tiff'}

print(f'Sigmas  : {list(SIGMAS)}')
print(f'Alpha   : {ALPHA}')
print(f'Beta    : {BETA}')
print(f'Gamma   : {GAMMA}')
print(f'Output  : {OUTPUT_DIR}')


# ================================================================
# CELL 5 — Collect all image paths
# ================================================================

# If exact path not found, fall back to full /kaggle/input search
if os.path.isdir(IMAGE_DIR):
    search_root = IMAGE_DIR
else:
    search_root = '/kaggle/input'
    print(f'⚠️  Path not found, searching: {search_root}')

all_image_paths = []
for dirpath, _, filenames in os.walk(search_root):
    for fname in filenames:
        if os.path.splitext(fname)[1].lower() in VALID_EXTENSIONS:
            all_image_paths.append(os.path.join(dirpath, fname))

all_image_paths.sort()
print(f'📦 Found {len(all_image_paths)} images')


# ================================================================
# CELL 6 — Run batch pipeline (same logic as your original code)
# ================================================================

failed  = []
t0      = time.time()

for idx, img_path in enumerate(all_image_paths):

    base = os.path.splitext(os.path.basename(img_path))[0]

    # ── Skip if already done (resume support) ──
    vessel_out = os.path.join(OUTPUT_DIR, f'{base}_vessel_map.png')
    if SAVE_VESSEL_MAP and os.path.exists(vessel_out):
        continue

    # ── Progress every 50 images ──
    if idx % 50 == 0:
        elapsed = time.time() - t0
        done    = idx + 1
        eta     = (elapsed / done) * (len(all_image_paths) - done) if done > 1 else 0
        print(f'[{done}/{len(all_image_paths)}]  {base}  '
              f'| elapsed {elapsed/60:.1f}min | ETA ~{eta/60:.1f}min')

    # ── Load ──
    image_bgr = cv2.imread(img_path)
    if image_bgr is None:
        print(f'  ⚠️  Cannot load: {img_path}')
        failed.append(img_path)
        continue

    # ── Pipeline ──
    try:
        results = vessel_enhancement_pipeline(
            image_bgr, sigmas=SIGMAS, alpha=ALPHA, beta=BETA, gamma=GAMMA
        )
    except Exception as e:
        print(f'  ❌ Error on {base}: {e}')
        failed.append(img_path)
        continue

    # ── Save individual files ──
    if SAVE_VESSEL_MAP:
        cv2.imwrite(
            os.path.join(OUTPUT_DIR, f'{base}_vessel_map.png'),
            (results['vessel_map'] * 255).astype(np.uint8)
        )
    if SAVE_GREEN:
        cv2.imwrite(
            os.path.join(OUTPUT_DIR, f'{base}_green.png'),
            results['green']
        )
    if SAVE_NORMALIZED:
        cv2.imwrite(
            os.path.join(OUTPUT_DIR, f'{base}_normalized.png'),
            (results['normalized'] * 255).astype(np.uint8)
        )
    if SAVE_FOV_MASK:
        cv2.imwrite(
            os.path.join(OUTPUT_DIR, f'{base}_fov_mask.png'),
            results['fov_mask']
        )

total_time = time.time() - t0
print(f'\n✅ Batch done! {len(all_image_paths) - len(failed)} processed in {total_time/60:.1f} min')
if failed:
    print(f'⚠️  {len(failed)} failed: {failed[:5]}')


# ================================================================
# CELL 7 — Visualize one sample result (optional check)
# ================================================================

sample_path = all_image_paths[0]
BASE        = os.path.splitext(os.path.basename(sample_path))[0]
image_bgr   = cv2.imread(sample_path)

results = vessel_enhancement_pipeline(
    image_bgr, sigmas=SIGMAS, alpha=ALPHA, beta=BETA, gamma=GAMMA
)

visualize_pipeline(
    results,
    title='Retinal Vessel Enhancement Pipeline — Sample',
    save_path=os.path.join(OUTPUT_DIR, f'{BASE}_pipeline.png')
)

# Side-by-side comparison
fig, axes = plt.subplots(1, 2, figsize=(14, 6))
fig.patch.set_facecolor('white')
axes[0].imshow(cv2.cvtColor(results['raw_bgr'], cv2.COLOR_BGR2RGB))
axes[0].set_title('Raw Fundus Image', fontsize=13, fontweight='bold')
axes[0].axis('off')
axes[1].imshow(results['vessel_map'], cmap='gray', vmin=0, vmax=1)
axes[1].set_title('Vessel Enhanced Map\n(white bg, black vessels)',
                   fontsize=13, fontweight='bold')
axes[1].axis('off')
plt.tight_layout()
plt.savefig(
    os.path.join(OUTPUT_DIR, f'{BASE}_comparison.png'),
    dpi=150, bbox_inches='tight', facecolor='white'
)
plt.show()
plt.close()


# ================================================================
# CELL 8 — Augmentation examples (same as original)
# ================================================================

N = 4
fig, axes = plt.subplots(2, N, figsize=(5 * N, 10))
fig.patch.set_facecolor('white')
fig.suptitle('Augmentation Examples', fontsize=14, fontweight='bold', y=1.01)

for i in range(N):
    aug_img, aug_map = augment_pair(
        results['normalized'],
        results['vessel_map']
    )
    axes[0, i].imshow(aug_img, cmap='gray', vmin=0, vmax=1)
    axes[0, i].set_title(f'Normalized aug {i+1}', fontsize=11)
    axes[0, i].axis('off')
    axes[1, i].imshow(aug_map, cmap='gray', vmin=0, vmax=1)
    axes[1, i].set_title(f'Vessel map aug {i+1}', fontsize=11)
    axes[1, i].axis('off')

plt.tight_layout()
plt.savefig(
    os.path.join(OUTPUT_DIR, f'{BASE}_augmentations.png'),
    dpi=120, bbox_inches='tight', facecolor='white'
)
plt.show()
plt.close()


# ================================================================
# CELL 9 — Save all outputs as ZIP (same as original Colab cell)
# ================================================================

zip_path = '/kaggle/working/vessel_outputs.zip'

with zipfile.ZipFile(zip_path, 'w', zipfile.ZIP_DEFLATED) as zf:
    for fname in sorted(os.listdir(OUTPUT_DIR)):
        full_path = os.path.join(OUTPUT_DIR, fname)
        zf.write(full_path, fname)   # stores only filename, not full path

zip_size_mb = os.path.getsize(zip_path) / (1024 * 1024)
print(f'✅ ZIP created: {zip_path}')
print(f'   Size : {zip_size_mb:.1f} MB')
print(f'   Files: {len(os.listdir(OUTPUT_DIR))}')

# Show saved files list
print('\nSaved files:')
for f in sorted(os.listdir(OUTPUT_DIR)):
    size = os.path.getsize(os.path.join(OUTPUT_DIR, f)) // 1024
    print(f'  {f}  ({size} KB)')